# DOT Line-Item Classification: Predicting Significant Budget Modifications

Builds a binary classifier that flags DOT budget line items likely to see a
**significant modification** (Modified far from Adopted) using only
information available at Adopted-budget time. Same DOT scope and cleaning
as `dot_analysis.ipynb` (`Agency == "Department of Transportation"`).

Each step prints its own output for verification. **Stops after Step 7** —
no further tuning yet.

In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    confusion_matrix,
    classification_report,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 130)

DATA_DIR = "raw datasets"
BUDGET_PATH = f"{DATA_DIR}/DOT_Budget_2017_2027.csv"


## Step 1: Load

Read the DOT budget file and apply the same `Agency == "Department of
Transportation"` filter used in `dot_analysis.ipynb`. No modeling
assumptions yet -- just load, filter, and look.

In [2]:
budget_raw = pd.read_csv(BUDGET_PATH)
print("Raw file shape:", budget_raw.shape)

budget = budget_raw[budget_raw["Agency"] == "Department of Transportation"].copy()
print("DOT-only row count:", len(budget), "(expected ~172K)")

print("\nColumns:", list(budget.columns))
print("\nDtypes:")
print(budget.dtypes)
print("\nHead:")
budget.head()


Raw file shape: (334806, 9)
DOT-only row count: 172576 (expected ~172K)

Columns: ['Adopted', 'Agency', 'Budget Code', 'Expense Category', 'Modified', 'Post Adjustments', 'Pre-Encumbered', 'Year', 'Source_File']

Dtypes:
Adopted             float64
Agency                  str
Budget Code             str
Expense Category        str
Modified            float64
Post Adjustments    float64
Pre-Encumbered      float64
Year                  int64
Source_File             str
dtype: object

Head:


,Adopted,Agency,Budget Code,Expense Category,Modified,Post Adjustments,Pre-Encumbered,Year,Source_File
0,56261392.0,Department of Transportation,4125,HEAT LIGHT & POWER,53742028.0,0.00,0.0,2017,nyc-data-feed DOT 2017.csv
1,39071937.0,Department of Transportation,2002,SUPPLIES + MATERIALS - GENERAL,42563150.0,120.94,0.0,2017,nyc-data-feed DOT 2017.csv
2,32955700.0,Department of Transportation,4122,MAINT & OPER OF INFRASTRUCTURE,30104225.0,0.00,0.0,2017,nyc-data-feed DOT 2017.csv
3,27844600.0,Department of Transportation,3100,FULL YEAR POSITIONS,29090761.0,0.00,0.0,2017,nyc-data-feed DOT 2017.csv
4,30256607.0,Department of Transportation,1270,RENTALS - LAND BLDGS & STRUCTS,25870624.0,0.00,0.0,2017,nyc-data-feed DOT 2017.csv


**Column-name check.** The requested feature list (`Object Class Name` /
`Object Code Name`, `Budget Code Name`, a `Personal Service/OTPS indicator`)
doesn't literally exist in this file's schema — confirmed above, the real
columns are `Adopted`, `Agency`, `Budget Code`, `Expense Category`,
`Modified`, `Post Adjustments`, `Pre-Encumbered`, `Year`, `Source_File`.
Mapping used below, so nothing is silently assumed:

| Requested | Used here | Why |
|---|---|---|
| Fiscal Year | `Year` | same concept, different column name |
| Object Class Name / Object Code Name | `Expense Category` | the descriptive line-item label (142 unique values for DOT) — closest match |
| Budget Code Name | `Budget Code` | there's no separate descriptive-name column for the code; the code itself (995 unique values) is used as the categorical identifier |
| Personal Service/OTPS indicator | `ps_otps_indicator` (derived) | no such column exists; derived from `Expense Category` text via a keyword rule — built and audited in Step 3 below |
| Adopted amount | `Adopted` | present as-is |

## Step 2: Build Target

```
modification    = Modified - Adopted
significant_mod = 1  if |modification| >= 0.10 * |Adopted| AND |modification| >= 50000
                 = 1  if Adopted == 0 AND |modification| >= 50000
                 = 0  otherwise
```

In [3]:
modification = budget["Modified"] - budget["Adopted"]
abs_mod = modification.abs()
abs_adopted = budget["Adopted"].abs()

cond_pct_and_floor = (abs_adopted > 0) & (abs_mod >= 0.10 * abs_adopted) & (abs_mod >= 50_000)
cond_zero_adopted = (budget["Adopted"] == 0) & (abs_mod >= 50_000)

budget["modification"] = modification
budget["significant_mod"] = (cond_pct_and_floor | cond_zero_adopted).astype(int)

print("Class balance (significant_mod):")
counts = budget["significant_mod"].value_counts().rename({0: "0 (stable)", 1: "1 (significant mod)"})
pct = budget["significant_mod"].value_counts(normalize=True).rename({0: "0 (stable)", 1: "1 (significant mod)"}) * 100
balance = pd.DataFrame({"count": counts, "pct": pct.round(2)})
print(balance)


Class balance (significant_mod):
                      count    pct
significant_mod                   
0 (stable)           167320  96.95
1 (significant mod)    5256   3.05


## Step 3: Features

Drop anything derived from `Modified` (`Modified` itself, `modification`,
and — not requested but also excluded to stay strictly to the 5 requested
features — `Post Adjustments` / `Pre-Encumbered`, which are separate raw
budget-modification components, not part of the requested feature set).

No literal `Personal Service/OTPS indicator` column exists (confirmed in
Step 1), so one is derived here from `Expense Category` text using a
keyword rule, audited below before use.

In [4]:
# --- Derive ps_otps_indicator from Expense Category text ---
# Explicit-tag overrides first (categories that literally say OTPS or -PS),
# then a keyword rule for personal-service-related line items (positions,
# overtime, differentials, leave, bonus/salary items, employee benefits),
# defaulting everything else to OTPS. This is a heuristic in the absence of
# a real PS/OTPS crosswalk column -- audited below, not applied blindly.

PS_KEYWORDS = [
    "POSITION", "OVERTIME", "DIFFERENTIAL", "UNSALARIED", "BACKPAY",
    "HOLIDAY PAY", "SUPPER MONEY", "SALARY", "TERMINAL LEAVE", "BONUS",
    "GROSS", "UNIFORM", "FRINGE BENEFIT", "PAYROLL", "HEALTH CLUB",
    "WELF BEN", "ANNUITY", "HEALTH INSURANCE", "SOCIAL SECURITY",
    "DISABILITY BENEFIT", "BENEFIC DECSD", "EMPLOYMENT SERVICES",
]

def classify_ps_otps(category: str) -> str:
    text = str(category).upper()
    if "OTPS" in text:
        return "OTPS"
    if "-PS" in text:
        return "PS"
    if any(kw in text for kw in PS_KEYWORDS):
        return "PS"
    return "OTPS"

budget["ps_otps_indicator"] = budget["Expense Category"].map(classify_ps_otps)

print("Derived ps_otps_indicator -- row counts:")
print(budget["ps_otps_indicator"].value_counts())
print()
print("Derived ps_otps_indicator -- distinct Expense Category values per bucket:")
print(budget.groupby("ps_otps_indicator")["Expense Category"].nunique())


Derived ps_otps_indicator -- row counts:
ps_otps_indicator
OTPS    93371
PS      79205
Name: count, dtype: int64

Derived ps_otps_indicator -- distinct Expense Category values per bucket:
ps_otps_indicator
OTPS    98
PS      44
Name: Expense Category, dtype: int64


In [5]:
# Audit: show which categories landed in each bucket so the rule can be
# checked at a glance, not trusted blindly.
print("Sample of categories classified as PS (up to 20):")
print(sorted(budget.loc[budget["ps_otps_indicator"] == "PS", "Expense Category"].unique())[:20])
print()
print("Sample of categories classified as OTPS (up to 20):")
print(sorted(budget.loc[budget["ps_otps_indicator"] == "OTPS", "Expense Category"].unique())[:20])


Sample of categories classified as PS (up to 20):
['ADD GROSS(& FRINGES) HOLD CODE', 'ALLOWANCE FOR UNIFORMS', 'AMOUNT TO BE SCHEDULED-PS', 'ANNUITY CONTRIBUTIONS', 'ASSIGNMENT DIFFERENTIAL', 'BACKPAY - PRIOR YEARS', 'BONUS PAYMENTS', 'BONUS- NONPENSIONABLE', 'DISABILITY BENEFITS INSURANCE', 'EARLY RET.TERMINAL LEAVE......', 'EDUC AND LICENCE DIFFERENTIAL', 'EMPLOYMENT SERVICES', 'FRINGE BENEFITS-OTHER', 'FULL TIME UNIFORMED PERSONNEL', 'FULL YEAR POSITIONS', 'HEALTH CLUB REIMBURSEMENT', 'HEALTH INSURANCE PLAN CITY EMP', 'HOLIDAY PAY', 'LONGEVITY DIFFERENTIAL', 'NEW POSITIONS']

Sample of categories classified as OTPS (up to 20):
['ADVERTISING', 'AUTOMOTIVE SUPPLIES & MATERIAL', 'AWARD TO BEN OF POLICE/FIREMEN', 'BAD DEBT EXPENSE', 'BALANCE UNASSIGNED-OTPS', 'BANK FEES', 'BOOKS-OTHER', 'CHARTER SCHOOLS', 'CHILDRENS CHARITABLE INSTITUTN', 'CLEANING SERVICES', 'CLEANING SUPPLIES', 'COMMUNITY CONSULTANT CONTRACTS', 'CONTRACTUAL SERVICES GENERAL', 'CONTRACTUAL SERVICES-GENERAL', 'COSTS ASS

In [6]:
# --- Assemble the feature set ---
FEATURE_COLUMNS = ["Year", "Expense Category", "Budget Code", "ps_otps_indicator", "Adopted"]
TARGET_COLUMN = "significant_mod"

LEAKY_COLUMNS = {"Modified", "modification", "Post Adjustments", "Pre-Encumbered"}
overlap = LEAKY_COLUMNS.intersection(FEATURE_COLUMNS)
print("Chosen features:", FEATURE_COLUMNS)
print("Leaky columns explicitly excluded:", sorted(LEAKY_COLUMNS))
print("Overlap between chosen features and leaky columns (should be empty):", overlap)
assert not overlap, "A leaky column ended up in the feature set!"

print("\nMissing values in chosen features:")
print(budget[FEATURE_COLUMNS].isna().sum())

X = budget[FEATURE_COLUMNS].copy()
y = budget[TARGET_COLUMN].copy()
print("\nX shape:", X.shape, " y shape:", y.shape)
X.head()


Chosen features: ['Year', 'Expense Category', 'Budget Code', 'ps_otps_indicator', 'Adopted']
Leaky columns explicitly excluded: ['Modified', 'Post Adjustments', 'Pre-Encumbered', 'modification']
Overlap between chosen features and leaky columns (should be empty): set()

Missing values in chosen features:
Year                 0
Expense Category     0
Budget Code          0
ps_otps_indicator    0
Adopted              0
dtype: int64

X shape: (172576, 5)  y shape: (172576,)


,Year,Expense Category,Budget Code,ps_otps_indicator,Adopted
0,2017,HEAT LIGHT & POWER,4125,OTPS,56261392.0
1,2017,SUPPLIES + MATERIALS - GENERAL,2002,OTPS,39071937.0
2,2017,MAINT & OPER OF INFRASTRUCTURE,4122,OTPS,32955700.0
3,2017,FULL YEAR POSITIONS,3100,PS,27844600.0
4,2017,RENTALS - LAND BLDGS & STRUCTS,1270,OTPS,30256607.0


## Step 4: Split

Time-based: train on Fiscal Year <= 2024, test on Fiscal Year 2025-2027.
No shuffling — this mirrors how the model would actually be used (predict
forward from past budgets).

In [7]:
train_mask = budget["Year"] <= 2024
test_mask = budget["Year"].between(2025, 2027)

X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[test_mask], y[test_mask]

print(f"Train (FY<=2024): {len(X_train):,} rows")
print(f"Test  (FY2025-2027): {len(X_test):,} rows")
print(f"Total: {len(X_train) + len(X_test):,} of {len(budget):,} (should match unless a Year fell outside both ranges)")

print("\nClass balance in TRAIN:")
print((y_train.value_counts(normalize=True) * 100).round(2))
print("\nClass balance in TEST:")
print((y_test.value_counts(normalize=True) * 100).round(2))


Train (FY<=2024): 123,002 rows
Test  (FY2025-2027): 49,574 rows
Total: 172,576 of 172,576 (should match unless a Year fell outside both ranges)

Class balance in TRAIN:
significant_mod
0    96.7
1     3.3
Name: proportion, dtype: float64

Class balance in TEST:
significant_mod
0    97.6
1     2.4
Name: proportion, dtype: float64


## Step 5: Model

`ColumnTransformer`: `OneHotEncoder` for `Expense Category`, `Budget Code`,
`ps_otps_indicator`; `StandardScaler` for the numeric features (`Year`,
`Adopted`). Baseline `LogisticRegression(class_weight="balanced")`, then
`RandomForestClassifier` (default class weighting, as specified) for
comparison.

In [8]:
CATEGORICAL_FEATURES = ["Expense Category", "Budget Code", "ps_otps_indicator"]
NUMERIC_FEATURES = ["Year", "Adopted"]

def build_preprocessor():
    return ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=True), CATEGORICAL_FEATURES),
            ("num", StandardScaler(), NUMERIC_FEATURES),
        ]
    )

pipe_lr = Pipeline(steps=[
    ("prep", build_preprocessor()),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
])

pipe_rf = Pipeline(steps=[
    ("prep", build_preprocessor()),
    ("clf", RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)),
])

print("Fitting LogisticRegression...")
pipe_lr.fit(X_train, y_train)
print("Done. Encoded feature count:", pipe_lr.named_steps["prep"].transform(X_train.iloc[:1]).shape[1])

print("\nFitting RandomForestClassifier...")
pipe_rf.fit(X_train, y_train)
print("Done.")


Fitting LogisticRegression...


Done. Encoded feature count: 1056

Fitting RandomForestClassifier...


Done.


## Step 6: Evaluate

Precision / recall / F1 / ROC-AUC on the FY2025-2027 test set, plus a
confusion matrix, for both models. Recall on class 1 (catching lines that
actually get modified) is the number to watch — missing a real
modification is the costlier error for a budget-monitoring tool than a
false alarm.

In [9]:
def evaluate(name, pipe, X_te, y_te):
    y_pred = pipe.predict(X_te)
    y_proba = pipe.predict_proba(X_te)[:, 1]

    print(f"=== {name} ===")
    print(classification_report(y_te, y_pred, target_names=["0 (stable)", "1 (significant mod)"], digits=3))
    print("ROC-AUC:", round(roc_auc_score(y_te, y_proba), 4))

    cm = confusion_matrix(y_te, y_pred)
    cm_df = pd.DataFrame(
        cm,
        index=["Actual 0 (stable)", "Actual 1 (sig. mod)"],
        columns=["Pred 0 (stable)", "Pred 1 (sig. mod)"],
    )
    print("\nConfusion matrix:")
    print(cm_df)

    recall_1 = cm[1, 1] / cm[1].sum() if cm[1].sum() > 0 else float("nan")
    print(f"\n>>> Recall on class 1 (significant_mod): {recall_1:.3f} "
          f"({cm[1,1]} of {cm[1].sum()} true modifications caught)")
    print()
    return {"y_pred": y_pred, "y_proba": y_proba, "cm": cm}

lr_results = evaluate("LogisticRegression (class_weight='balanced')", pipe_lr, X_test, y_test)


=== LogisticRegression (class_weight='balanced') ===
                     precision    recall  f1-score   support

         0 (stable)      0.993     0.863     0.924     48382
1 (significant mod)      0.121     0.761     0.208      1192

           accuracy                          0.861     49574
          macro avg      0.557     0.812     0.566     49574
       weighted avg      0.972     0.861     0.907     49574

ROC-AUC: 0.9032

Confusion matrix:
                     Pred 0 (stable)  Pred 1 (sig. mod)
Actual 0 (stable)              41774               6608
Actual 1 (sig. mod)              285                907

>>> Recall on class 1 (significant_mod): 0.761 (907 of 1192 true modifications caught)



In [10]:
rf_results = evaluate("RandomForestClassifier (default class weighting)", pipe_rf, X_test, y_test)


=== RandomForestClassifier (default class weighting) ===
                     precision    recall  f1-score   support

         0 (stable)      0.987     0.983     0.985     48382
1 (significant mod)      0.406     0.462     0.432      1192

           accuracy                          0.971     49574
          macro avg      0.696     0.723     0.709     49574
       weighted avg      0.973     0.971     0.972     49574

ROC-AUC: 0.8846

Confusion matrix:
                     Pred 0 (stable)  Pred 1 (sig. mod)
Actual 0 (stable)              47576                806
Actual 1 (sig. mod)              641                551

>>> Recall on class 1 (significant_mod): 0.462 (551 of 1192 true modifications caught)



## Step 7: Feature Importance (Random Forest)

Top 15 one-hot-encoded features by Gini importance.

In [11]:
feature_names = pipe_rf.named_steps["prep"].get_feature_names_out()
importances = pipe_rf.named_steps["clf"].feature_importances_

importance_df = (
    pd.DataFrame({"feature": feature_names, "importance": importances})
    .sort_values("importance", ascending=False)
    .head(15)
    .reset_index(drop=True)
)
print("Top 15 features driving significant_mod (Random Forest):")
importance_df


Top 15 features driving significant_mod (Random Forest):


,feature,importance
0,num__Year,0.243459
1,num__Adopted,0.240402
2,cat__Expense Category_FULL YEAR POSITIONS,0.013100
3,cat__Budget Code_Z030,0.012336
4,cat__ps_otps_indicator_OTPS,0.010166
5,cat__ps_otps_indicator_PS,0.009408
6,cat__Budget Code_2118,0.009113
7,cat__Expense Category_EQUIPMENT GENERAL,0.007894
8,cat__Expense Category_SUPPLIES + MATERIALS - G...,0.007583
9,cat__Expense Category_CONTRACTUAL SERVICES GEN...,0.007430


---
**Stopping here per instructions.** Both models are trained and evaluated
on the FY2025-2027 hold-out; metrics are printed above (Step 6) and Random
Forest feature importances above (Step 7). No further tuning yet.